# 5교시. 문서 자동화 웹 애플리케이션 기본 구현

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/05_streamlit_basic.ipynb)

**이번 교시 행동:** 업로드·실행·원문·JSON 영역을 만들고 업로드한 파일명이 화면에 반영되는지 확인합니다.

**통과 증거:** `course_outputs/app_05.py`

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

화면의 실행 모드를 먼저 확인합니다.

- `LIVE`: 현재 파일에 실제 모델을 실행한 결과
- `PREPARED_FALLBACK`: 공개 샘플을 사람이 검수해 둔 복구 결과
- 3분 이상 멈추면 실행을 중지하고 복구 결과로 계속합니다.
- 각 교시 끝에서 `CHECKPOINT PASS`와 산출물 파일을 확인합니다.


In [ ]:
import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
VALIDATION_MODE = os.getenv("COURSE_VALIDATE_PREPARED") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or VALIDATION_MODE:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 준비 입력을 쓰려면 "
            "USE_PREPARED_INPUT=True로 바꾸세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if VALIDATION_MODE:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())


In [ ]:
import importlib.metadata
import subprocess

required_streamlit = "1.60.0"
try:
    installed_streamlit = importlib.metadata.version("streamlit")
except importlib.metadata.PackageNotFoundError:
    installed_streamlit = None
if installed_streamlit != required_streamlit:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", f"streamlit=={required_streamlit}"]
    )


In [ ]:
from textwrap import dedent

app_code = 'import streamlit as st\n\nGOLDEN_RECEIPT = {\'document_type\': \'receipt\', \'store_name\': \'이태리집\', \'date\': \'2025-10-04\', \'total_amount\': 76000, \'items\': [{\'name\': \'페퍼로니 앤 치즈\', \'quantity\': 1, \'unit_price\': 29000, \'line_total\': 29000}, {\'name\': \'토마토 파스타\', \'quantity\': 1, \'unit_price\': 14000, \'line_total\': 14000}, {\'name\': \'수제 돈가스\', \'quantity\': 1, \'unit_price\': 13000, \'line_total\': 13000}, {\'name\': \'새우 칠리치 필라\', \'quantity\': 1, \'unit_price\': 14000, \'line_total\': 14000}, {\'name\': \'콜라\', \'quantity\': 3, \'unit_price\': 2000, \'line_total\': 6000}], \'adjustments\': {\'discount\': 0, \'tax\': 0, \'service\': 0, \'rounding\': 0}, \'tax_breakdown\': {\'mode\': \'included_in_item_prices\', \'supply_amount\': 69094, \'vat\': 6906, \'payable_total\': 76000}, \'raw_values\': {\'store_name\': \'이태리집\', \'date\': \'2025-10-04 12:33:37\', \'total_amount\': \'76,000\'}, \'cleaned_values\': {\'store_name\': \'이태리집\', \'date\': \'2025-10-04\', \'total_amount\': 76000}, \'evidence\': {\'store_name\': {\'raw_value\': \'이태리집\', \'line\': 1}, \'date\': {\'raw_value\': \'거래일시 2025-10-04 12:33:37\', \'line\': 2}, \'total_amount\': {\'raw_value\': \'합계 금액 76,000\', \'line\': 8}}, \'source_mode\': \'prepared_fixture_rule_extraction\'}\nGOLDEN_OCR_TEXT = \'이태리집\\n거래일시 2025-10-04 12:33:37\\n페퍼로니 앤 치즈 29,000 1 29,000\\n토마토 파스타 14,000 1 14,000\\n수제 돈가스 13,000 1 13,000\\n새우 칠리치 필라 14,000 1 14,000\\n콜라 2,000 3 6,000\\n합계 금액 76,000\\n부가세 과세물품가액 69,094\\n부가세 6,906\\n\'\n\nst.set_page_config(page_title="영수증 Document AI", layout="wide")\nst.title("영수증 Document AI 미니 앱")\nuploaded = st.file_uploader(\n    "승인된 비식별 이미지 또는 PDF 한 장 · 최대 5MB",\n    type=["png", "jpg", "jpeg", "pdf"],\n    max_upload_size=5,\n    help="PNG, JPG, JPEG, PDF만 허용합니다. 수업에서는 한 번에 5MB 이하 한 장만 처리합니다.",\n)\nif uploaded is not None:\n    st.success(f"업로드 연결 확인: {uploaded.name} · {len(uploaded.getvalue()):,} bytes")\n    st.caption("이 파일은 6교시에서 실제 처리 함수와 연결합니다.")\n\nif st.button("공개 샘플 준비 결과 보기"):\n    st.info("실행 모드: PREPARED_FALLBACK")\n    st.text_area("판독 원문", GOLDEN_OCR_TEXT, height=220)\n    st.json(GOLDEN_RECEIPT)\n'
output_path = OUTPUT_DIR / "app_05.py"
output_path.write_text(app_code, encoding="utf-8")
print("저장:", output_path)


In [ ]:
from streamlit.testing.v1 import AppTest

app_test = AppTest.from_file(str(output_path)).run(timeout=20)
assert not app_test.exception
assert app_test.title[0].value == "영수증 Document AI 미니 앱"
assert len(app_test.file_uploader) == 1
assert len(app_test.button) == 1
app_test.button[0].click().run(timeout=20)
assert any("PREPARED_FALLBACK" in item.value for item in app_test.info)
print("CHECKPOINT 1/1 PASS: 업로드·버튼·결과 화면")
